# Decision Table Module Demo

This notebook demonstrates:
1. Creating decision table configurations with parameters, expressions, and outputs
2. Using DeciderBuilder to expand nodes from decision table configuration
3. Executing decision table logic with sample data

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import polars as pl
from decider.modules.decision_table.config import (
    DecisionTable,
    DecisionTableConfig,
    ParametersConfig,
    AndExpression,
    OrExpression,
    BetweenExpression,
    InExpression,
    IsTrueExpression
)
from decider.modules.decision_table.impl import (
    calculate_decision_table_output,
    evaluate_decision_table_from_config,
    extract_struct_fields
)

## Step 1: Create Sample Data

In [ ]:
# Create sample loan application data
data = {
    "application_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_age": [25, 35, 45, 22, 50, 60, 28, 40],
    "annual_income": [45000, 75000, 90000, 35000, 120000, 85000, 40000, 95000],
    "employment_type": ["full_time", "full_time", "contract", "part_time", "full_time", "retired", "unemployed", "full_time"],
    "credit_score": [650, 720, 780, 580, 800, 750, 520, 760],
    "loan_amount": [200000, 350000, 450000, 150000, 500000, 300000, 100000, 400000]
}

lf = pl.LazyFrame(data)
print("Sample loan application data:")
lf.collect()

## Step 2: Define Decision Table Configuration

In [ ]:
# Define loan approval decision table
config_data = {
    "parameters": {
        "columns": ["age_min", "age_max", "income_min", "valid_employment", "credit_min", "decision", "interest_rate", "max_loan"],
        "values": [
            # Young professionals with good credit
            [25, 35, 50000, ["full_time", "contract"], 650, "approve", 3.5, 400000],
            # Mid-career with excellent credit
            [30, 50, 70000, ["full_time", "contract"], 700, "approve", 3.2, 600000], 
            # Senior professionals
            [45, 65, 80000, ["full_time", "contract", "retired"], 680, "approve", 3.8, 500000],
            # High earners (any age)
            [18, 70, 100000, ["full_time", "contract"], 600, "approve", 3.0, 800000],
            # Conservative approval for lower income
            [25, 45, 40000, ["full_time"], 700, "conditional", 4.2, 250000]
        ],
        "dtypes": {
            "age_min": "float",
            "age_max": "float", 
            "income_min": "float",
            "valid_employment": "list[string]",
            "credit_min": "float",
            "decision": "string",
            "interest_rate": "float",
            "max_loan": "float"
        }
    },
    "expression": {
        "typ": "and",
        "expressions": [
            {
                "typ": "between",
                "variable": "age",
                "lower_bound_column": "age_min", 
                "upper_bound_column": "age_max",
                "mode": "both_inclusive"
            },
            {
                "typ": "between",
                "variable": "income",
                "lower_bound_column": "income_min",
                "upper_bound_column": None,
                "mode": "lower_inclusive"
            },
            {
                "typ": "in",
                "variable": "employment",
                "values_column": "valid_employment"
            },
            {
                "typ": "between",
                "variable": "credit",
                "lower_bound_column": "credit_min",
                "upper_bound_column": None,
                "mode": "lower_inclusive"
            }
        ]
    },
    "outputs": ["decision", "interest_rate", "max_loan"],
    "default": ["reject", 5.0, 0]
}

# Create decision table configuration
dt_config = DecisionTableConfig(**config_data)
print("✓ Decision table configuration created")
print(f"Parameters shape: {dt_config.parameters._parameters_df.shape}")
print(f"Output columns: {dt_config.outputs}")
print(f"Default values: {dt_config.default}")

## Step 3: Test Standalone Implementation

In [ ]:
# Test the standalone implementation with sample inputs
print("Testing standalone implementation:")

# Test case 1: Young professional - should match first rule
test_cases = [
    {"name": "Young Professional", "age": 28, "income": 55000, "employment": "full_time", "credit": 680},
    {"name": "Mid-career Executive", "age": 40, "income": 85000, "employment": "full_time", "credit": 720},
    {"name": "High Earner", "age": 35, "income": 120000, "employment": "contract", "credit": 650},
    {"name": "Rejection Case", "age": 22, "income": 30000, "employment": "part_time", "credit": 550},
]

for i, case in enumerate(test_cases):
    result_expr = evaluate_decision_table_from_config(
        dt_config,
        age=pl.lit(case["age"]),
        income=pl.lit(case["income"]),
        employment=pl.lit(case["employment"]),
        credit=pl.lit(case["credit"])
    )
    
    # Evaluate the expression
    test_df = pl.DataFrame({"test_id": [i]})
    result_df = test_df.with_columns(result_expr.alias("loan_decision"))
    
    result = result_df["loan_decision"][0]
    print(f"{case['name']}: {result}")

## Step 4: Create Decision Table Module

In [ ]:
# Create the decision table module
decision_table = DecisionTable(
    config=dt_config,
    output_name="loan_approval_result"
)

print(f"✓ Decision table module created: {decision_table}")
print(f"Output name: {decision_table.output_name}")
print(f"Configuration type: {type(decision_table.config)}")

## Step 5: Test Node Expansion

In [ ]:
# Test node expansion
print("Testing node expansion:")
try:
    nodes = decision_table.expand_nodes({})
    print(f"✓ Nodes created: {list(nodes.keys())}")
    
    # Inspect the created node
    node = nodes[decision_table.output_name]
    print(f"Node name: {node.name}")
    print(f"Node input types: {list(node.input_types.keys())}")
    print(f"Node callable: {node.callable}")
    
except Exception as e:
    print(f"Error expanding nodes: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Build DeciderBuilder and Execute

In [ ]:
from decider.dag.builder import DeciderBuilder

# Build the DeciderBuilder with our decision table
try:
    dr = (
        DeciderBuilder()
        .with_config({})
        .include(decision_table)
        .build()
    )
    
    print(f"✓ DeciderBuilder created successfully: {dr}")
    print(f"Available nodes: {list(dr.graph.nodes.keys())}")
    
except Exception as e:
    print(f"Error building DeciderBuilder: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Execute the decision table with loan application data
if 'dr' in locals():
    try:
        result = dr.execute(
            ["loan_approval_result"],  # Output we want
            inputs={
                "age": pl.col("customer_age"),
                "income": pl.col("annual_income"),
                "employment": pl.col("employment_type"),
                "credit": pl.col("credit_score")
            }
        )
        
        print("✓ Decision table execution result:")
        print(result["loan_approval_result"])
        
    except Exception as e:
        print(f"Error executing decision table: {e}")
        import traceback
        traceback.print_exc()
else:
    print("DeciderBuilder not available for execution")

## Step 7: Apply to DataFrame with Results Extraction

In [ ]:
# Apply decision table logic directly to our sample data
print("Applying decision table to DataFrame:")

# Create DataFrame with our loan applications
df = lf.collect()

# Apply decision logic using our configuration
result_expr = evaluate_decision_table_from_config(
    dt_config,
    age=pl.col("customer_age"),
    income=pl.col("annual_income"),
    employment=pl.col("employment_type"),
    credit=pl.col("credit_score")
)

# Add decision results to DataFrame
df_with_decisions = df.with_columns(result_expr.alias("loan_decision"))

print("✓ DataFrame with decision results:")
print(df_with_decisions.select(["application_id", "customer_age", "annual_income", "employment_type", "credit_score", "loan_decision"]))

In [ ]:
# Extract individual decision fields
print("Extracting decision fields:")

df_extracted = extract_struct_fields(
    df_with_decisions,
    "loan_decision",
    ["decision", "interest_rate", "max_loan"]
)

print("✓ DataFrame with extracted fields:")
print(df_extracted.select(["application_id", "customer_age", "annual_income", "decision", "interest_rate", "max_loan"]))

## Step 8: Test Complex Expression Logic

In [ ]:
# Create a more complex decision table with OR logic
complex_config_data = {
    "parameters": {
        "columns": ["condition_type", "threshold", "categories", "result"],
        "values": [
            ["high_income", 100000, [], "premium_customer"],
            ["good_credit", 750, [], "standard_customer"],
            ["special_employment", 0, ["doctor", "lawyer", "engineer"], "professional_customer"]
        ],
        "dtypes": {
            "condition_type": "string",
            "threshold": "float",
            "categories": "list[string]", 
            "result": "string"
        }
    },
    "expression": {
        "typ": "or",
        "expressions": [
            {
                "typ": "between",
                "variable": "income",
                "lower_bound_column": "threshold", 
                "upper_bound_column": None,
                "mode": "lower_inclusive"
            },
            {
                "typ": "between",
                "variable": "credit",
                "lower_bound_column": "threshold",
                "upper_bound_column": None,
                "mode": "lower_inclusive"
            },
            {
                "typ": "in",
                "variable": "job_category",
                "values_column": "categories"
            }
        ]
    },
    "outputs": ["result"],
    "default": ["basic_customer"]
}

complex_config = DecisionTableConfig(**complex_config_data)
print("✓ Complex decision table with OR logic created")

# Test with sample data
test_result = evaluate_decision_table_from_config(
    complex_config,
    income=pl.lit(120000),  # High income - should match first rule
    credit=pl.lit(650),
    job_category=pl.lit("teacher")
)

test_df = pl.DataFrame({"test": [1]})
result_df = test_df.with_columns(test_result.alias("customer_type"))
print(f"High income test: {result_df['customer_type'][0]}")

# Test professional category
test_result2 = evaluate_decision_table_from_config(
    complex_config,
    income=pl.lit(50000),
    credit=pl.lit(650),
    job_category=pl.lit("doctor")  # Professional - should match third rule
)

result_df2 = test_df.with_columns(test_result2.alias("customer_type"))
print(f"Professional test: {result_df2['customer_type'][0]}")

## Summary

This notebook demonstrated:

1. **Decision Table Configuration**: Created complex decision tables with multiple parameter types (ranges, lists, thresholds)
2. **Expression Logic**: Tested AND and OR expressions with various condition types (between, in)
3. **Standalone Implementation**: Used implementation functions directly for testing and validation
4. **Node Expansion**: Successfully expanded decision table configuration into Hamilton nodes
5. **DeciderBuilder Integration**: Built execution graphs from decision table modules
6. **DataFrame Processing**: Applied decision logic to real datasets with result extraction

The decision table module successfully:
- Validated complex tabular decision logic with Pandera
- Generated efficient Polars expressions for decision evaluation  
- Integrated with the Hamilton-based decision framework
- Supported flexible output structures and default handling